# 03 — Validate

Runs the validation battery from [methodology.md](methodology.md) §5a-5d on the fits saved by `02_Fit_Models`. Independently runnable as long as `data/results/` is populated.

**Tests executed:**
- **§5a (i)** Walk-forward hold-out CV (re-fits on truncated pre-period)
- **§5a (ii)** Moment matching (pre-period treated vs synthetic)
- **§5b** Pre-period parallel-fit defence (gap series stats)
- **§5c** Pre-period regime stability (distance correlation across pre-period thirds)
- **§5d** Donor SUTVA / cleanliness battery (bootstrap break + Wilcoxon event-window per donor)

**Outputs**: `data/validation/{event}_{test}.csv`

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from lib.config import T0, PRE_WINDOWS, MODEL_HPARAMS, DONOR_POOL_VARIANT
from lib.data import build_panel, load_fit, list_fits, save_validation_table
from lib.validation import (walk_forward_cv, parallel_fit_test, moment_matching,
                            regime_stability_test, event_window_return_test,
                            chow_break_test_bootstrap, benjamini_hochberg)

MODELS_AVAILABLE = ['convex_scm', 'ascm', 'elastic_net', 'xgboost', 'bsts']
EVENTS = ['russia', 'hormuz']
WINDOW = 'preferred'   # validate the preferred specification only
VARIANT = DONOR_POOL_VARIANT

print(f'Validating event = {EVENTS}, window = {WINDOW}, variant = {VARIANT}')
print(f'Saved fits available: {len(list_fits())}')

Validating event = ['russia', 'hormuz'], window = preferred, variant = shared
Saved fits available: 30


## §5a (i) — Walk-forward hold-out CV

Fits each model on the first 80% of the pre-period and evaluates on the last 20%. A model whose validation RMSE is much larger than its train RMSE is overfitting and gets a `passes=False` flag.

In [2]:
from lib.validation import get_tuned_hparams

wf_rows = []
for event in EVENTS:
    panel, meta = build_panel(event=event, window=WINDOW, variant=VARIANT)
    for model in MODELS_AVAILABLE:
        # Use the val-tuned hyperparameters that 02_Fit_Models stored with each fit,
        # falling back to config defaults for any keys not tuned (e.g. n_random_v for SCM).
        kwargs = {**MODEL_HPARAMS.get(model, {}),
                  **get_tuned_hparams(model, event, WINDOW, VARIANT)}
        try:
            r = walk_forward_cv(model, panel, 'Brent', meta['donors'],
                                t0=meta['t0'], t_pre_start=meta['t_pre_start'], **kwargs)
            r.update({'event': event, 'window': WINDOW})
            wf_rows.append(r)
        except Exception as e:
            wf_rows.append({'event': event, 'window': WINDOW, 'model': model,
                            'error': str(e)[:60]})

wf_df = pd.DataFrame(wf_rows)
save_validation_table(wf_df, 'walk_forward_cv')
wf_df.round(4)

,model,train_rmse,val_rmse,val_train_ratio,n_train,n_val,passes,event,window
0,convex_scm,0.1115,0.0858,0.7701,336,85,True,russia,preferred
1,ascm,0.0457,0.0872,1.9094,336,85,True,russia,preferred
2,elastic_net,0.0553,0.0939,1.6987,336,85,True,russia,preferred
3,xgboost,0.0380,0.1323,3.4805,336,85,False,russia,preferred
4,bsts,0.0335,0.1710,5.0991,336,85,False,russia,preferred
5,convex_scm,0.0589,0.1287,2.1848,338,85,False,hormuz,preferred
6,ascm,0.0349,0.0872,2.4963,338,85,False,hormuz,preferred
7,elastic_net,0.0431,0.0487,1.1307,338,85,True,hormuz,preferred
8,xgboost,0.0451,0.0991,2.1954,338,85,False,hormuz,preferred
9,bsts,0.0325,0.0934,2.8728,338,85,False,hormuz,preferred


## §5b — Pre-period parallel-fit defence

Tests on the pre-period gap series. A model fails if mean ≠ 0, AR(1) > 0.9, slope > 5%/year, or R² > 0.10.

In [3]:
pf_rows = []
for event in EVENTS:
    for model in MODELS_AVAILABLE:
        fit = load_fit(event, WINDOW, model, variant=VARIANT)
        if fit is None:
            continue
        r = parallel_fit_test(fit)
        r.update({'event': event, 'window': WINDOW, 'model': model})
        pf_rows.append(r)

pf_df = pd.DataFrame(pf_rows)
save_validation_table(pf_df, 'parallel_fit_defence')
pf_df.round(4)

,n,mean_pct,sd_pct,t_stat,p_mean_zero,ar1,slope_pct_per_year,p_slope_zero,r_squared,passes,event,window,model
0,421,0.4285,10.4786,0.8392,0.4019,0.9795,10.0104,0.0000,0.2097,False,russia,preferred,convex_scm
1,421,0.1140,4.8075,0.4866,0.6268,0.8811,0.3351,0.4942,0.0011,True,russia,preferred,ascm
2,421,0.1738,5.9482,0.5996,0.5491,0.9167,1.1874,0.0498,0.0092,False,russia,preferred,elastic_net
3,421,0.0740,3.8324,0.3960,0.6923,0.8463,3.0758,0.0000,0.1480,False,russia,preferred,xgboost
4,421,0.0932,4.3252,0.4422,0.6586,0.8517,0.1519,0.7305,0.0003,True,russia,preferred,bsts
5,423,0.2611,6.7584,0.7945,0.4273,0.9589,-7.0425,0.0000,0.2535,False,hormuz,preferred,convex_scm
6,423,0.0668,3.6638,0.3748,0.7080,0.8225,-0.1337,0.7178,0.0003,True,hormuz,preferred,ascm
7,423,0.0995,4.4997,0.4546,0.6496,0.9010,-0.5833,0.1986,0.0039,False,hormuz,preferred,elastic_net
8,423,0.0955,4.3669,0.4497,0.6532,0.9052,-3.2836,0.0000,0.1320,False,hormuz,preferred,xgboost
9,423,0.0603,3.4840,0.3558,0.7221,0.7744,-0.0534,0.8794,0.0001,True,hormuz,preferred,bsts


## §5a (ii) — Moment matching

Pre-period mean, SD, min, max, AR(1) of log-Brent vs log-synthetic. Δ Mean ≈ 0 is required; |Δ SD| > 25% of treated SD indicates the donor pool cannot span Brent's volatility.

In [4]:
for event in EVENTS:
    panel, meta = build_panel(event=event, window=WINDOW, variant=VARIANT)
    for model in MODELS_AVAILABLE:
        fit = load_fit(event, WINDOW, model, variant=VARIANT)
        if fit is None:
            continue
        df = moment_matching(fit, panel, 'Brent')
        df['event'] = event
        df['model'] = model
        save_validation_table(df, f'moments_{event}_{model}')
        print(f'\n{event} / {model}:')
        print(df.round(4).to_string())


russia / convex_scm:
      treated   synth   delta  delta_pct   event       model
mean   4.1281  4.1294  0.0013     0.0315  russia  convex_scm
sd     0.2693  0.2071 -0.0622   -23.0891  russia  convex_scm
min    3.5926  3.7625  0.1699     4.7287  russia  convex_scm
max    4.6216  4.4769 -0.1448    -3.1322  russia  convex_scm
ar1    0.9969  0.9983  0.0014     0.1383  russia  convex_scm

russia / ascm:
      treated   synth   delta  delta_pct   event model
mean   4.1281  4.1281  0.0000     0.0001  russia  ascm
sd     0.2693  0.2634 -0.0059    -2.1930  russia  ascm
min    3.5926  3.6817  0.0890     2.4780  russia  ascm
max    4.6216  4.5642 -0.0574    -1.2419  russia  ascm
ar1    0.9969  0.9971  0.0003     0.0262  russia  ascm

russia / elastic_net:
      treated   synth   delta  delta_pct   event        model
mean   4.1281  4.1281 -0.0000    -0.0000  russia  elastic_net
sd     0.2693  0.2573 -0.0120    -4.4499  russia  elastic_net
min    3.5926  3.6393  0.0466     1.2983  russia  elastic


hormuz / xgboost:
      treated   synth   delta  delta_pct   event    model
mean   4.2743  4.2743 -0.0000    -0.0003  hormuz  xgboost
sd     0.0923  0.0644 -0.0279   -30.2216  hormuz  xgboost
min    4.0932  4.1920  0.0988     2.4138  hormuz  xgboost
max    4.4848  4.3878 -0.0970    -2.1637  hormuz  xgboost
ar1    0.9792  0.9964  0.0172     1.7575  hormuz  xgboost

hormuz / bsts:
      treated   synth   delta  delta_pct   event model
mean   4.2743  4.2743  0.0000     0.0000  hormuz  bsts
sd     0.0923  0.0848 -0.0075    -8.0963  hormuz  bsts
min    4.0932  4.1164  0.0232     0.5673  hormuz  bsts
max    4.4848  4.4607 -0.0241    -0.5366  hormuz  bsts
ar1    0.9792  0.9875  0.0083     0.8470  hormuz  bsts


## §5c — Pre-period regime stability

For each event, computes Brent-donor distance correlation in three sub-periods of the pre-window. Donors whose distance correlation shifts by > 0.20 between thirds are flagged.

In [5]:
for event in EVENTS:
    panel, meta = build_panel(event=event, window=WINDOW, variant=VARIANT)
    df = regime_stability_test(panel, 'Brent', meta['donors'],
                               meta['t_pre_start'], meta['t_pre_end'])
    df['event'] = event
    save_validation_table(df, f'regime_stability_{event}')
    print(f'\n{event} regime stability (max_shift > 0.2 = flagged):')
    flagged = df[~df['stable']]
    print(f'  donors flagged: {len(flagged)} of {len(df)}')
    if len(flagged) > 0:
        print(flagged.round(3).to_string())


russia regime stability (max_shift > 0.2 = flagged):


  donors flagged: 14 of 21
          third_1  third_2  third_3  max_shift  stable   event
Platinum    0.845    0.559    0.600      0.286   False  russia
Coffee      0.598    0.848    0.560      0.288   False  russia
Cotton      0.713    0.543    0.796      0.253   False  russia
SP500       0.761    0.818    0.278      0.540   False  russia
WorldEq     0.796    0.815    0.468      0.348   False  russia
Nikkei      0.808    0.430    0.467      0.377   False  russia
AUD         0.821    0.599    0.334      0.488   False  russia
CHF         0.789    0.420    0.208      0.581   False  russia
CNY         0.679    0.385    0.601      0.294   False  russia
ZAR         0.736    0.585    0.366      0.370   False  russia
MXN         0.699    0.332    0.438      0.367   False  russia
TLT         0.612    0.486    0.848      0.362   False  russia
HYG         0.799    0.535    0.731      0.264   False  russia
VIX         0.697    0.598    0.402      0.295   False  russia

hormuz regime stability (m

## §5d — Donor SUTVA / cleanliness battery (statistical counterpart to donor catalog)

For each donor in the pool, run two model-free tests around the event date:
1. **Bootstrap structural break** — does the donor's log-return process shift at $T_0$?
2. **Wilcoxon event-window return** — is the donor's mean return in [-5d, +20d] different from a control window?

Apply Benjamini-Hochberg FDR correction at α = 0.10 across donors. A donor failing ≥ 1 of 2 (after FDR correction) is flagged for that event.

*(In-space placebo per donor — the third SUTVA test from methodology §5d — is implemented in 04_Inference because it's also used for treated-unit inference; results from 04 can be merged into this scorecard.)*

In [6]:
from lib.data import load_donors
donors_all = load_donors()

for event in EVENTS:
    panel, meta = build_panel(event=event, window=WINDOW, variant=VARIANT)
    t0 = meta['t0']
    rows = []
    for d in meta['donors']:
        s = donors_all[d].dropna()
        break_r = chow_break_test_bootstrap(s.loc[meta['t_pre_start']:meta['t_post_end']],
                                             break_date=t0)
        ew_r = event_window_return_test(s, event_date=t0)
        rows.append({
            'donor': d,
            'break_p': break_r.get('p', np.nan),
            'break_stat': break_r.get('observed_stat', np.nan),
            'ew_p': ew_r.get('p', np.nan),
            'ew_mean_window': ew_r.get('mean_window', np.nan),
            'ew_mean_control': ew_r.get('mean_control', np.nan),
        })
    df = pd.DataFrame(rows).set_index('donor')
    df['break_reject'] = benjamini_hochberg(df['break_p'].values, alpha=0.10)
    df['ew_reject'] = benjamini_hochberg(df['ew_p'].values, alpha=0.10)
    df['n_fail'] = df['break_reject'].astype(int) + df['ew_reject'].astype(int)
    df['flagged'] = df['n_fail'] >= 1   # any-fail trigger; methodology says ≥ 2 of 4 once all tests run
    save_validation_table(df, f'sutva_battery_{event}')
    print(f'\n{event} SUTVA scorecard (donors with n_fail ≥ 1):')
    print(df[df['flagged']].round(4).to_string())


russia SUTVA scorecard (donors with n_fail ≥ 1):
       break_p  break_stat    ew_p  ew_mean_window  ew_mean_control  break_reject  ew_reject  n_fail  flagged
donor                                                                                                        
JPY      0.006      0.2395  0.2123          0.0018           0.0001          True      False       1     True
CNY      0.000      0.3676  0.4442          0.0002          -0.0001          True      False       1     True



hormuz SUTVA scorecard (donors with n_fail ≥ 1):
        break_p  break_stat    ew_p  ew_mean_window  ew_mean_control  break_reject  ew_reject  n_fail  flagged
donor                                                                                                         
Cotton    0.004      0.3709  0.9959         -0.0001           0.0001          True      False       1     True


## Validation summary — pass/fail by (event, model)

In [7]:
summary_rows = []
for event in EVENTS:
    for model in MODELS_AVAILABLE:
        wf = wf_df[(wf_df.get('event') == event) & (wf_df.get('model') == model)]
        pf = pf_df[(pf_df.get('event') == event) & (pf_df.get('model') == model)]
        summary_rows.append({
            'event': event, 'model': model,
            'wf_train_rmse': wf['train_rmse'].iloc[0] if len(wf) else np.nan,
            'wf_val_rmse':   wf['val_rmse'].iloc[0]   if len(wf) else np.nan,
            'wf_passes':     bool(wf['passes'].iloc[0]) if len(wf) and 'passes' in wf else False,
            'pf_mean_pct':   pf['mean_pct'].iloc[0] if len(pf) else np.nan,
            'pf_slope_yr':   pf['slope_pct_per_year'].iloc[0] if len(pf) else np.nan,
            'pf_passes':     bool(pf['passes'].iloc[0]) if len(pf) and 'passes' in pf else False,
            'overall_pass':  bool((wf['passes'].iloc[0] if len(wf) and 'passes' in wf else False) and 
                                  (pf['passes'].iloc[0] if len(pf) and 'passes' in pf else False)),
        })
summary_df = pd.DataFrame(summary_rows)
save_validation_table(summary_df, 'validation_summary')
summary_df.round(4)

,event,model,wf_train_rmse,wf_val_rmse,wf_passes,pf_mean_pct,pf_slope_yr,pf_passes,overall_pass
0,russia,convex_scm,0.1115,0.0858,True,0.4285,10.0104,False,False
1,russia,ascm,0.0457,0.0872,True,0.1140,0.3351,True,True
2,russia,elastic_net,0.0553,0.0939,True,0.1738,1.1874,False,False
3,russia,xgboost,0.0380,0.1323,False,0.0740,3.0758,False,False
4,russia,bsts,0.0335,0.1710,False,0.0932,0.1519,True,False
5,hormuz,convex_scm,0.0589,0.1287,False,0.2611,-7.0425,False,False
6,hormuz,ascm,0.0349,0.0872,False,0.0668,-0.1337,True,False
7,hormuz,elastic_net,0.0431,0.0487,True,0.0995,-0.5833,False,False
8,hormuz,xgboost,0.0451,0.0991,False,0.0955,-3.2836,False,False
9,hormuz,bsts,0.0325,0.0934,False,0.0603,-0.0534,True,False
